# 01 Data Preparation

This notebook prepares the BODMAS data for training our two classifiers.

By the end, we want:

- `X_train`, `X_test`: cleaned feature data
- `y_train`, `y_test`: malware category labels, including `benign`
- summary tables we can use later in the report

Important: the assignment says to use malware categories from `bodmas_malware_category.csv`, not the family names from `bodmas_metadata.csv`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

BENIGN_LABEL = "benign"
TEST_SIZE = 0.2
RANDOM_STATE = 42

# This lets the notebook work whether it is run from the repo root or notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FEATURES_PATH = PROJECT_ROOT / "data" / "bodmas.npz"
METADATA_PATH = PROJECT_ROOT / "data" / "bodmas_metadata.csv"
CATEGORIES_PATH = PROJECT_ROOT / "data" / "bodmas_malware_category.csv"

FEATURES_PATH, METADATA_PATH, CATEGORIES_PATH

(WindowsPath('c:/Users/tomla/My Drive/Macquarie Uni/COMP8325 – Applications of Artificial Intelligence for Cyber Security/COMP8325-Assignment2-Group9/data/bodmas.npz'),
 WindowsPath('c:/Users/tomla/My Drive/Macquarie Uni/COMP8325 – Applications of Artificial Intelligence for Cyber Security/COMP8325-Assignment2-Group9/data/bodmas_metadata.csv'),
 WindowsPath('c:/Users/tomla/My Drive/Macquarie Uni/COMP8325 – Applications of Artificial Intelligence for Cyber Security/COMP8325-Assignment2-Group9/data/bodmas_malware_category.csv'))

## Load Raw Files

The BODMAS files give us features, binary benign/malware labels, SHA hashes, and malware categories.

In [2]:
# bodmas.npz contains the actual ML inputs.
npz = np.load(FEATURES_PATH)

# X = feature table. Each row is one program. Each column is one extracted feature.
X_raw = npz["X"]

# binary_y = original label from BODMAS. 0 = benign, 1 = malware.
# This is NOT our final assignment label, because we need malware categories.
binary_y = npz["y"]

# metadata has the SHA hash for each row in X.
metadata = pd.read_csv(METADATA_PATH)

# categories maps malware SHA hashes to labels like trojan, worm, ransomware, etc.
categories = pd.read_csv(CATEGORIES_PATH)

print(f"X shape: {X_raw.shape}")
print(f"binary_y length: {len(binary_y):,}")
print(f"metadata rows: {len(metadata):,}")
print(f"category rows: {len(categories):,}")

X shape: (134435, 2381)
binary_y length: 134,435
metadata rows: 134,435
category rows: 57,293


## Check File Alignment

The assignment says row `i` in `bodmas.npz` matches row `i` in `bodmas_metadata.csv`. If that is wrong, every label after this would be wrong.

In [3]:
# Row 0 in X must match row 0 in metadata, row 1 must match row 1, etc.
if len(metadata) != len(X_raw):
    raise ValueError(f"Feature/metadata row mismatch: X has {len(X_raw)} rows but metadata has {len(metadata)} rows.")

# We also need one binary label for every feature row.
if len(binary_y) != len(X_raw):
    raise ValueError(f"Feature/label row mismatch: X has {len(X_raw)} rows but binary_y has {len(binary_y)} rows.")

# We only check the columns we actually need.
# Important: metadata["family"] exists, but the assignment says NOT to use it.
required_metadata_cols = {"sha"}
required_category_cols = {"sha256", "category"}

if not required_metadata_cols.issubset(metadata.columns):
    raise ValueError(f"Metadata is missing required columns: {required_metadata_cols - set(metadata.columns)}")

if not required_category_cols.issubset(categories.columns):
    raise ValueError(f"Category file is missing required columns: {required_category_cols - set(categories.columns)}")

print("Alignment checks passed.")

Alignment checks passed.


## Build Final Labels

Malware rows get category labels from `bodmas_malware_category.csv`. Benign rows are labelled manually as `benign`.

In [4]:
# Make a lookup table like:
# "abc123..." -> "trojan"
# "def456..." -> "worm"
category_by_sha = categories.set_index("sha256")["category"]

# For each row in metadata, look up the SHA in the malware category table.
# Malware rows should get a category. Benign rows will be NaN for now.
labels = metadata["sha"].map(category_by_sha)

# Benign files are not listed in bodmas_malware_category.csv.
# So if binary_y is 0, we manually label that row as "benign".
labels = labels.where(binary_y == 1, BENIGN_LABEL).rename("label")

# This is just for checking/debugging.
# The model only needs X and labels, but this table lets us inspect rows later.
sample_index = pd.DataFrame({
    "sha": metadata["sha"],
    "binary_label": binary_y,
    "label": labels,
})

# If a malware row has no category, we cannot use it for category training.
# In our current data this number is 0, which is good.
missing_category_count = int(sample_index.loc[sample_index["binary_label"].eq(1), "label"].isna().sum())

print(f"Missing malware category labels: {missing_category_count:,}")
sample_index.head()

Missing malware category labels: 0


,sha,binary_label,label
0,e6d7b4bab32def853ab564410df53fa33172dda1bfd48c...,0,benign
1,5af37a058a5bcf2284c183ee98d92b7c66d8f5ce623e92...,0,benign
2,5bfbbea150af5cef2d3a93b80ef7c7faea9f564b56045d...,0,benign
3,216f592f1e1717d5681b7f5f2b14a28a2f0c603b5b7318...,0,benign
4,a1ca76813d2e9e7e23b830c87fbe29bcb51fcbe096e445...,0,benign


## Dataset Summary

These numbers tell us what we are training on and show class imbalance.

In [5]:
# Count how many samples are in each class.
# dropna=False means missing labels would show up instead of being hidden.
class_counts = labels.value_counts(dropna=False).rename_axis("label").reset_index(name="count")

dataset_summary = {
    "total_samples": int(X_raw.shape[0]),
    "total_features": int(X_raw.shape[1]),
    "total_classes": int(labels.dropna().nunique()),
    "benign_samples": int(labels.eq(BENIGN_LABEL).sum()),
    "malware_samples": int(sample_index["binary_label"].eq(1).sum()),
    "missing_malware_category_labels": missing_category_count,
}

for key, value in dataset_summary.items():
    print(f"{key}: {value:,}")

class_counts

total_samples: 134,435
total_features: 2,381
total_classes: 15
benign_samples: 77,142
malware_samples: 57,293
missing_malware_category_labels: 0


,label,count
0,benign,77142
1,trojan,29972
2,worm,16697
3,backdoor,7331
4,downloader,1031
5,ransomware,821
6,dropper,715
7,informationstealer,448
8,virus,192
9,pua,29


## Remove Rows Without Labels

Any row with no final label cannot be used for supervised training. Our current data should keep every row.

In [6]:
# labels.notna() is True for rows that have a real class label.
labelled_rows = labels.notna().to_numpy()

X_labelled = X_raw[labelled_rows]
y = labels.loc[labelled_rows].reset_index(drop=True)

print(f"Rows before: {len(labels):,}")
print(f"Rows after: {len(y):,}")

Rows before: 134,435
Rows after: 134,435


## Check Feature Values

Decision trees and random forests still need normal numeric values. This checks for bad numbers and useless constant columns.

In [7]:
# NaN means "not a number". Most sklearn models cannot train with NaN values.
nan_values = int(np.isnan(X_labelled).sum())

# Infinite values are also invalid for normal sklearn training.
positive_inf_values = int(np.isposinf(X_labelled).sum())
negative_inf_values = int(np.isneginf(X_labelled).sum())

# A constant feature has the same value for every sample.
# It cannot help the model separate classes, so we can safely remove it.
feature_min_before = np.nanmin(X_labelled, axis=0)
feature_max_before = np.nanmax(X_labelled, axis=0)
constant_features_before = int(np.sum(feature_min_before == feature_max_before))

feature_checks_before = {
    "nan_values": nan_values,
    "positive_inf_values": positive_inf_values,
    "negative_inf_values": negative_inf_values,
    "constant_features": constant_features_before,
}

feature_checks_before

{'nan_values': 0,
 'positive_inf_values': 0,
 'negative_inf_values': 0,
 'constant_features': 49}

## Clean Features

This replaces invalid numbers if they exist, then removes columns that never change.

In [8]:
# This keeps the code safe if bad values appear later.
# In our current dataset, these replacements should not change anything.
X_finite = np.nan_to_num(X_labelled, nan=0.0, posinf=0.0, neginf=0.0)

# True means "keep this feature".
# False means "drop this feature because it never changes".
feature_min_after_fixing = np.min(X_finite, axis=0)
feature_max_after_fixing = np.max(X_finite, axis=0)
kept_feature_mask = feature_min_after_fixing != feature_max_after_fixing

X = X_finite[:, kept_feature_mask]

# Check the cleaned feature matrix.
feature_checks_after = {
    "nan_values": int(np.isnan(X).sum()),
    "positive_inf_values": int(np.isposinf(X).sum()),
    "negative_inf_values": int(np.isneginf(X).sum()),
    "constant_features": int(np.sum(np.min(X, axis=0) == np.max(X, axis=0))),
}

print(f"Original features: {X_raw.shape[1]:,}")
print(f"Features kept: {int(kept_feature_mask.sum()):,}")
print(f"Features removed: {int((~kept_feature_mask).sum()):,}")
feature_checks_after

Original features: 2,381
Features kept: 2,332
Features removed: 49


{'nan_values': 0,
 'positive_inf_values': 0,
 'negative_inf_values': 0,
 'constant_features': 0}

## Check Rare Classes

Stratified splitting needs each class to have at least two samples. Otherwise, one class cannot appear in both train and test.

In [9]:
# Stratified splitting tries to keep each class represented in train and test.
# A class with fewer than 2 samples cannot be split into both sets.
rare_classes = class_counts.loc[class_counts["count"] < 2].reset_index(drop=True)

if not rare_classes.empty:
    raise ValueError(f"Some classes are too small for stratified splitting: {rare_classes.to_dict(orient='records')}")

print("All classes have enough samples for stratified splitting.")

All classes have enough samples for stratified splitting.


## Train/Test Split

We use an 80/20 stratified split so the train and test sets have roughly the same class proportions.

In [10]:
# stratify=y keeps the class proportions roughly the same in train and test.
# This matters because the dataset is very imbalanced.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Reset indexes so y_train/y_test are clean Series from 0..n.
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {len(y_train):,}")
print(f"y_test: {len(y_test):,}")

X_train: (107548, 2332)
X_test: (26887, 2332)
y_train: 107,548
y_test: 26,887


## Check Split Class Counts

This makes sure every class appears in both train and test.

In [11]:
train_counts = y_train.value_counts().rename_axis("label").reset_index(name="train_count")
test_counts = y_test.value_counts().rename_axis("label").reset_index(name="test_count")

split_class_counts = class_counts.merge(train_counts, on="label", how="left").merge(test_counts, on="label", how="left")
split_class_counts["train_count"] = split_class_counts["train_count"].fillna(0).astype(int)
split_class_counts["test_count"] = split_class_counts["test_count"].fillna(0).astype(int)

if (split_class_counts["train_count"] == 0).any() or (split_class_counts["test_count"] == 0).any():
    raise ValueError("At least one class is missing from train or test.")

split_class_counts

,label,count,train_count,test_count
0,benign,77142,61714,15428
1,trojan,29972,23977,5995
2,worm,16697,13358,3339
3,backdoor,7331,5865,1466
4,downloader,1031,825,206
5,ransomware,821,657,164
6,dropper,715,572,143
7,informationstealer,448,358,90
8,virus,192,153,39
9,pua,29,23,6


## Save Local Summary Outputs

These CSV files are for local checking and report drafting. They are ignored by git because the repo should not store generated data outputs.

In [12]:
metrics_dir = PROJECT_ROOT / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

class_counts.to_csv(metrics_dir / "class_distribution.csv", index=False)
split_class_counts.to_csv(metrics_dir / "split_class_distribution.csv", index=False)
pd.DataFrame([dataset_summary]).to_csv(metrics_dir / "dataset_summary.csv", index=False)
pd.DataFrame([feature_checks_before]).to_csv(metrics_dir / "feature_checks_before.csv", index=False)
pd.DataFrame([feature_checks_after]).to_csv(metrics_dir / "feature_checks_after.csv", index=False)

split_summary = pd.DataFrame([
    {"split": "train", "samples": len(y_train), "features": X_train.shape[1]},
    {"split": "test", "samples": len(y_test), "features": X_test.shape[1]},
])
split_summary.to_csv(metrics_dir / "split_summary.csv", index=False)

print(f"Wrote summary files to {metrics_dir}")

Wrote summary files to c:\Users\tomla\My Drive\Macquarie Uni\COMP8325 – Applications of Artificial Intelligence for Cyber Security\COMP8325-Assignment2-Group9\results\metrics
